## Exercise 7 - Testing Tableau

In [1]:
import geopandas as gpd
import intake
import pandas as pd


import calitp
from calitp.tables import tbl
from calitp import query_sql
from siuba import *

import shapely
from shapely.geometry import Polygon, LineString, Point
from shapely import wkt
from shapely.geometry import Point, Polygon
from shapely.wkt import loads

import shared_utils 
import os
from calitp.tables import tbl

os.environ["CALITP_BQ_MAX_BYTES"] = str(50_000_000_000)

WGS84 = "EPSG:4326"
CA_StatePlane = "EPSG:2229"  # units are in feet
CA_NAD83Albers = "EPSG:3310"  # units are in meters

SQ_MI_PER_SQ_M = 3.86 * 10 ** -7
FEET_PER_MI = 5_280
SQ_FT_PER_SQ_MI = 2.788 * 10 ** 7


/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.10.2-CAPI-1.16.0) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(
E0329 15:18:55.143816796     956 fork_posix.cc:70]           Fork support is only compatible with the epoll1 and poll polling strategies


In [41]:
#vars(shared_utils)

## A few Bay Area agencies
[Agencies.yml](https://github.com/cal-itp/data-infra/blob/main/airflow/data/agencies.yml)
* AC Transit: 4
* BART: 279
* SF Muni: 282 
* Samtrans: 290
*  Santa Clara Valley Transportation Authority: 294

In [3]:
#choosing a different set of ITP IDS
ITP_ID = [4,279,282,290,294]

In [4]:
bay_area = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

/opt/conda/lib/python3.9/site-packages/siuba/sql/utils.py:52: SAWarning: Dialect bigquery:bigquery will not make use of SQL compilation caching as it does not set the 'supports_statement_cache' attribute to ``True``.  This can have significant performance implications including some performance degradations in comparison to prior SQLAlchemy versions.  Dialect maintainers should seek to set this attribute to True after appropriate development and testing for SQLAlchemy 1.4 caching support.   Alternatively, this attribute may be set to False which will disable this warning. (Background on this error at: https://sqlalche.me/e/14/cprf)


In [5]:
bay_area.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name
0,4,10,37.742485,-122.250108,Aughinbaugh Way & Robert Davey Jr Dr
1,4,100,37.759356,-122.251237,2217 Otis Dr


In [6]:
f'A total of {len(bay_area)} rows'

'A total of 25283 rows'

In [7]:
bay_area2 = bay_area.drop_duplicates(subset=['stop_id','itp_id'])

In [8]:
f'only {len(bay_area2)} rows after dropping duplicates'

'only 25026 rows after dropping duplicates'

In [9]:
#creating point geometry from longtitude &  latitude 
bay_area2 = shared_utils.geography_utils.create_point_geometry(bay_area2, 
                                                               'stop_lon', 'stop_lat')

In [10]:
bay_area2.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry
0,4,10,37.742485,-122.250108,Aughinbaugh Way & Robert Davey Jr Dr,POINT (-122.25011 37.74249)
1,4,100,37.759356,-122.251237,2217 Otis Dr,POINT (-122.25124 37.75936)


In [11]:
def make_routes_shapefile(ITP_ID_LIST=[], CRS="EPSG:4326", alternate_df=None):
    """
    Parameters:
    ITP_ID_LIST: list. List of ITP IDs found in agencies.yml
    CRS: str. Default is WGS84, but able to re-project to another CRS.
    Returns a geopandas.GeoDataFrame, where each line is the operator-route-line geometry.
    """

    all_routes = gpd.GeoDataFrame()

    for itp_id in ITP_ID_LIST:
        if alternate_df is None:
            shapes = (
                tbl.gtfs_schedule.shapes()
                >> filter(_.calitp_itp_id == int(itp_id))
                >> collect()
            )

        elif alternate_df is not None:
            shapes = alternate_df.copy()

            # With historical gtfs_schedule_type2.shapes table, there is a valid shape_id
            if shapes.shape_id.isnull().sum() == 0:
                pass
            else:
                # shape_id is None, which will throw up an error later on when there's groupby
                # So, create shape_id and fill it in with route_id info, but only if it's missing
                # To flag these, shape_id as a column wouldn't even exist.
                if "shape_id" not in shapes.columns:
                    shapes = shapes.assign(
                        shape_id=shapes.route_id,
                    )
                else:
                    # This handles operators where most of their shape_ids are valid
                    # Let's just drop missing ones.
                    shapes = shapes[shapes.shape_id.notna()].reset_index(drop=True)

        # Make a gdf
        shapes = gpd.GeoDataFrame(
            shapes,
            geometry=gpd.points_from_xy(shapes.shape_pt_lon, shapes.shape_pt_lat),
            crs=WGS84,
        )

        # Count the number of stops for a given shape_id
        # Must be > 1 (need at least 2 points to make a line)
        shapes = shapes.assign(
            num_stops=(
                shapes.groupby("shape_id")["shape_pt_sequence"].transform("count")
            )
        )

        # Drop the shape_ids that can't make a line
        shapes = shapes[shapes.num_stops > 1].reset_index(drop=True)

        # Now, combine all the stops by stop sequence, and create linestring
        for route in shapes.shape_id.unique():
            single_shape = (
                shapes
                >> filter(_.shape_id == route)
                >> mutate(shape_pt_sequence=_.shape_pt_sequence.astype(int))
                # arrange in the order of stop sequence
                >> arrange(_.shape_pt_sequence)
            )

            # Convert from a bunch of points to a line (for a route, there are multiple points)
            route_line = shapely.geometry.LineString(list(single_shape["geometry"]))
            single_route = single_shape[
                ["calitp_itp_id", "calitp_url_number", "shape_id"]
            ].iloc[
                [0]
            ]  # preserve info cols
            single_route["geometry"] = route_line
            single_route = gpd.GeoDataFrame(single_route, crs=WGS84)

            # https://stackoverflow.com/questions/15819050/pandas-dataframe-concat-vs-append/48168086
            all_routes = pd.concat(
                [all_routes, single_route], ignore_index=True, axis=0
            )

    all_routes = (
        all_routes.to_crs(CRS)
        .sort_values(["calitp_itp_id", "shape_id"])
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return all_routes


In [12]:
#bay_area3 = make_routes_shapefile(ITP_ID_LIST = [4,279,282,290,294], CRS="EPSG:4326")

In [13]:
#bay_area3.head()

In [14]:
#bay_area3.to_file('bay_area_line.geojson', driver='GeoJSON')  

## San Francisco Muni Only
* Get dim stops & dim times into one df.
* [Resource 1](https://github.com/cal-itp/data-analyses/blob/main/bus_service_increase/warehouse_queries.py#L71-L76)
* [Resource 2](https://github.com/cal-itp/data-analyses/blob/main/bus_service_increase/setup_parallel_trips_with_stops.py#L136-L153)
    * What df are you supposed to put in here?
* [Resource 3](https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html)


In [15]:
#just getting stops table first.
SF = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id == 282)
    >> distinct()
)

In [16]:
SF.shape

(6373, 5)

In [17]:
SF = SF.drop_duplicates(subset = ['stop_id','stop_name'])


In [18]:
SF.stop_id.nunique()

6328

In [44]:
SF.sort_values('stop_id').head(50)

,itp_id,stop_id,stop_lat,stop_lon,stop_name
0,282,10390,37.721190,-122.475096,19th Avenue & Holloway St
1,282,10913,37.719192,-122.425802,Dublin St & La Grande Ave
2,282,13016,37.772618,-122.389786,3rd St & 4th St
3,282,13018,37.727739,-122.403240,Bacon St & San Bruno Ave
4,282,13019,37.727645,-122.403269,Bacon St & San Bruno Ave
5,282,13020,37.726670,-122.407460,Bacon St & Somerset St
6,282,13021,37.726540,-122.407660,Bacon St & Somerset St
7,282,13023,37.790249,-122.482263,Baker Beach Parking Lot Sw
8,282,13024,37.797606,-122.445693,Baker St & Greenwich St
9,282,13031,37.806094,-122.409161,Bay St & Midway St


### Stop Sequence
* Why do I need to get calitp deleted at > max date


In [20]:
'''
Tiffany's Code.
    tbl_stop_times = (tbl.views.gtfs_schedule_dim_stop_times()
                      >> filter(_.calitp_extracted_at <= min_date, 
                                _.calitp_deleted_at > max_date)
                      >> select(_.calitp_itp_id, _.trip_id, _.departure_time,
                                _.stop_sequence, _.stop_id)
    )
   
'''

"\nTiffany's Code.\n    tbl_stop_times = (tbl.views.gtfs_schedule_dim_stop_times()\n                      >> filter(_.calitp_extracted_at <= min_date, \n                                _.calitp_deleted_at > max_date)\n                      >> select(_.calitp_itp_id, _.trip_id, _.departure_time,\n                                _.stop_sequence, _.stop_id)\n    )\n   \n"

In [21]:
SELECTED_DATE = "2022-02-28"


In [22]:
#getting times so I can grab the sequence of stops.
SF_times = (
    tbl.views.gtfs_schedule_stop_times()
    >> filter(_.calitp_itp_id == 282)
    >> filter(_.calitp_extracted_at <= SELECTED_DATE)
    >> select(_.calitp_itp_id, _.trip_id, _.departure_time,
                                _.stop_sequence, _.stop_id)
    >> collect()
    >> distinct()
)

In [23]:
SF_times.dtypes

calitp_itp_id      int64
trip_id           object
departure_time    object
stop_sequence      int64
stop_id           object
dtype: object

In [24]:
#filter out for only one trip id, 9987569
#https://www.sfmta.com/routes/29-sunset
SF_times2 = SF_times.loc[SF_times['trip_id'] == '9987569'] 

In [25]:
SF_times2.stop_id.nunique()

92

In [26]:
SF_times2.shape

(165, 5)

### How do go about the stop sequence corresponding with different stop ids?
* 21, 22, 23, 24 are examples

In [27]:
SF_times2.sort_values('stop_sequence').head(50)

,calitp_itp_id,trip_id,departure_time,stop_sequence,stop_id
20624,282,9987569,13:36:00,1,4648
1983191,282,9987569,13:36:36,2,4647
1751045,282,9987569,13:37:21,3,4646
1989597,282,9987569,13:38:09,4,4645
972095,282,9987569,13:39:10,5,4784
381960,282,9987569,13:39:45,6,5955
692532,282,9987569,13:40:52,7,5115
1523742,282,9987569,13:41:35,8,5116
399915,282,9987569,13:42:14,9,4976
2226693,282,9987569,13:42:58,10,4785


### Merging

In [28]:
SF_times2 = SF_times2.drop(columns=['calitp_itp_id'])

In [29]:
SF_joined = pd.merge(SF_times2, SF,  how='inner', on=['stop_id'], indicator=True)

In [30]:
SF_joined._merge.value_counts()

both          165
left_only       0
right_only      0
Name: _merge, dtype: int64

In [31]:
SF_joined.shape

(165, 9)

In [32]:
SF_joined2 = shared_utils.geography_utils.create_point_geometry(SF_joined, 
                                                               'stop_lon', 'stop_lat')

* Again, stop sequence and stop names sometimes don't match starting at sequence 21.
* Persia & Prague St appears at stop sequence 27 and 28.

In [33]:
SF_joined2.sort_values('stop_sequence').head(50)

,trip_id,departure_time,stop_sequence,stop_id,itp_id,stop_lat,stop_lon,stop_name,_merge,geometry
14,9987569,13:36:00,1,4648,282,37.722895,-122.394842,Fitzgerald Ave & Keith St,both,POINT (-122.39484 37.72290)
156,9987569,13:36:36,2,4647,282,37.722028,-122.393320,Fitzgerald Ave & Jennings St,both,POINT (-122.39332 37.72203)
149,9987569,13:37:21,3,4646,282,37.720966,-122.391453,Fitzgerald Ave & Ingalls St,both,POINT (-122.39145 37.72097)
157,9987569,13:38:09,4,4645,282,37.719921,-122.389587,Fitzgerald Ave & Hawes St,both,POINT (-122.38959 37.71992)
103,9987569,13:39:10,5,4784,282,37.718234,-122.388272,Gilman Ave & Griffith St,both,POINT (-122.38827 37.71823)
51,9987569,13:39:45,6,5955,282,37.717502,-122.386990,Gilman Ave and Giants Dr,both,POINT (-122.38699 37.71750)
88,9987569,13:40:52,7,5115,282,37.717012,-122.389212,Ingerson Ave & Griffith St,both,POINT (-122.38921 37.71701)
138,9987569,13:41:35,8,5116,282,37.718039,-122.390998,Ingerson Ave & Hawes St,both,POINT (-122.39100 37.71804)
52,9987569,13:42:14,9,4976,282,37.719198,-122.390012,Hawes St & Gilman Ave,both,POINT (-122.39001 37.71920)
162,9987569,13:42:58,10,4785,282,37.720350,-122.391729,Gilman Ave & Ingalls St,both,POINT (-122.39173 37.72035)


In [34]:
SF_joined2.stop_sequence.nunique()

93

In [35]:
len(SF_joined2)

165

* After dropping duplicate stop IDs, now some of the sequences are out of order/missing.

In [36]:
SF_joined3 = SF_joined2.drop_duplicates(subset=['stop_id'])

In [43]:
SF_joined3.sort_values('stop_sequence').head(50)

,trip_id,departure_time,stop_sequence,stop_id,itp_id,stop_lat,stop_lon,stop_name,_merge,geometry
14,9987569,13:36:00,1,4648,282,37.722895,-122.394842,Fitzgerald Ave & Keith St,both,POINT (-122.39484 37.72290)
156,9987569,13:36:36,2,4647,282,37.722028,-122.393320,Fitzgerald Ave & Jennings St,both,POINT (-122.39332 37.72203)
149,9987569,13:37:21,3,4646,282,37.720966,-122.391453,Fitzgerald Ave & Ingalls St,both,POINT (-122.39145 37.72097)
157,9987569,13:38:09,4,4645,282,37.719921,-122.389587,Fitzgerald Ave & Hawes St,both,POINT (-122.38959 37.71992)
103,9987569,13:39:10,5,4784,282,37.718234,-122.388272,Gilman Ave & Griffith St,both,POINT (-122.38827 37.71823)
51,9987569,13:39:45,6,5955,282,37.717502,-122.386990,Gilman Ave and Giants Dr,both,POINT (-122.38699 37.71750)
88,9987569,13:40:52,7,5115,282,37.717012,-122.389212,Ingerson Ave & Griffith St,both,POINT (-122.38921 37.71701)
138,9987569,13:41:35,8,5116,282,37.718039,-122.390998,Ingerson Ave & Hawes St,both,POINT (-122.39100 37.71804)
52,9987569,13:42:14,9,4976,282,37.719198,-122.390012,Hawes St & Gilman Ave,both,POINT (-122.39001 37.71920)
162,9987569,13:42:58,10,4785,282,37.720350,-122.391729,Gilman Ave & Ingalls St,both,POINT (-122.39173 37.72035)


In [38]:
#SF_joined4.sort_values('stop_sequence').head()

In [39]:
#SF_joined4.to_file('SF_joined4.geojson', driver='GeoJSON')  